# LangChain Agent Benchmark 01: Models, Prompts, Context, and Outputs

This notebook benchmarks model and prompt variables while keeping the task constant. It uses OpenRouter chat completions with free model variants.

How to use it: for every model's response, follow the reflections points to give a 1-5 rating for each model.
At the end of the notebook, sum your answers to see how the models fare on these biological tasks.

## OpenRouter free-model limits

Rate limits govern how many requests you can make. There are a few rate limits that apply to certain types of requests, regardless of account status:

**Free usage limits**: When using a free model variant (with an ID ending in `:free`),as we are in this tutorial, the following limits apply:

| Credits purchased (all time) | Requests per minute | Requests per day |
|---|---:|---:|
| Less than 10 | 20 | 50 |
| At least 10 | 20 | 1000 |

Keep an eye on your usage here: https://openrouter.ai/activity/

**DDoS protection**: Cloudflare's DDoS protection will block requests that dramatically exceed reasonable usage.

See OpenRouter's limits documentation: https://openrouter.ai/docs/api_reference/limits#handling-429-errors


## 0. Library import and global variable definition

Before running this code cell, make sure you have set up your `.env` file with `OPENROUTER_API_KEY`. This contains your personal credentials and is read with `os.getenv()`.

In [41]:
from dotenv import load_dotenv
import os
import time
from typing import Literal

import pandas as pd
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv("../.env", override=True)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_SITE_URL = os.getenv("OPENROUTER_SITE_URL")
OPENROUTER_APP_NAME = os.getenv("OPENROUTER_APP_NAME", "LangChain Agent Benchmark")

# Free OpenRouter models allow 20 requests/minute. A small pause keeps full-notebook runs below that limit.
REQUEST_PAUSE_SECONDS = 3

MODEL_SPECS = [
    {
        "model_type": "OpenAI GPT-OSS 20B free model",
        "model": "openai/gpt-oss-20b:free",
    },
    {
        "model_type": "NVIDIA Nemotron Nano Omni free reasoning model",
        "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    },
    {
        "model_type": "Google Gemma 4 31B: instruction tuned, dense",
        "model": "google/gemma-4-31b-it:free",
    },
    {
        "model_type": "Google Gemma 4 26B A4B, instruction model, MoE",
        "model": "google/gemma-4-26b-a4b-it:free",
    },
]
DEFAULT_MODEL = MODEL_SPECS[0]
DEFAULT_MODEL_ID = DEFAULT_MODEL["model"]

if OPENROUTER_API_KEY:
    print("OpenRouter API key loaded.")
else:
    print("Set OPENROUTER_API_KEY before running the examples.")


OpenRouter API key loaded.


In [42]:
default_headers = {"X-Title": OPENROUTER_APP_NAME}
if OPENROUTER_SITE_URL:
    default_headers["HTTP-Referer"] = OPENROUTER_SITE_URL

OPENROUTER_CLIENT = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    default_headers=default_headers,
)


## 1. LLM choice

LLM choice affects factual accuracy, biological knowledge, reasoning quality, hallucination behavior, speed, verbosity, and code quality. This section isolates several model-level variables one at a time where the available free OpenRouter models allow it.


### Different vendors

Different labs make different modeling choices: training data, instruction tuning, safety behavior, output style, and evaluation targets. This block keeps the task constant and compares one free model from OpenAI, NVIDIA, and Google.

**Reflection Prompts**
- Compare which answers seem most reliable and identify which biological details make you think so.
- Separate content quality from style: is a more fluent answer also more correct?
- Note whether one vendor consistently gives more caveats, stronger claims, or clearer uncertainty.


In [ ]:
question = 'Explain why batch correction matters in single-cell RNA-seq analysis. Keep the answer under 120 words.'
vendor_models = [
    {"vendor": "OpenAI", "model": "openai/gpt-oss-20b:free"},
    {"vendor": "NVIDIA", "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"},
    {"vendor": "Google", "model": "google/gemma-4-31b-it:free"},
]

for spec in vendor_models:
    heading = spec['vendor']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### Number of parameters

This block keeps the comparison inside the Nemotron family so the parameter-size discussion is less confounded by vendor. It compares a small Nemotron model, a Nano Omni model, and a larger Super model. The models still differ in architecture and modality support, so treat this as a practical size comparison rather than a perfectly controlled ablation.

**Reflection Prompts**
- Does the larger model give a more specific or more cautious biological answer?
- Compare latency and token use against answer quality: is the larger model worth it here?
- Look for whether extra scale improves scientific precision or mainly changes style.


In [ ]:
question = 'Interpret high FKBP5 and CRISPLD2 expression after dexamethasone treatment. Keep the answer under 120 words.'
parameter_models = [
    {"parameter_label": "9B", "model": "nvidia/nemotron-nano-9b-v2:free"},
    # {"parameter_label": "30B stored, ~3B active", "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"},
    {"parameter_label": "120B stored, ~12B active", "model": "nvidia/nemotron-3-super-120b-a12b:free"},
]

for spec in parameter_models:
    heading = spec['parameter_label']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### Different LLM architecture

A dense model uses all of its stored parameters for every token, making behavior more direct but often more compute-heavy. A Mixture of Experts (MoE) model routes each token through a subset of parameters (called "experts"), which can reduce compute and latency but may change reproducibility and style. This block compares regular Gemma 4 against its MoE counterpart.

**Reflection Prompts**
- Does dense versus MoE change answer quality, or mostly latency and token behavior?
- Look for instability or oddly different framing between architectures.
- Decide which architecture you would choose for a repeated scientific-note workflow.


In [ ]:
question = 'A T-cell cluster has high interferon-stimulated genes after stimulation. Explain the biological interpretation and two checks. Keep the answer under 120 words.'
architecture_models = [
    {"architecture": "dense", "model": "google/gemma-4-31b-it:free"},
    {"architecture": "MoE", "model": "google/gemma-4-26b-a4b-it:free"},
]

for spec in architecture_models:
    heading = spec['architecture']
    chunks = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": question}],
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices and chunk.choices[0].delta.content:
            chunks.append(chunk.choices[0].delta.content)
            markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
            answer_display.update(Markdown(markdown_text))
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### Reasoning vs non-reasoning

Reasoning models spend extra internal tokens before producing the final answer. This is one of the biggest conceptual gaps in current LLM use because it can change latency, token use, and accuracy.

Use OpenRouter's `reasoning` request parameter. With the OpenAI Python SDK, safest is to pass it through `extra_body`.

- `reasoning.enabled: True` enables reasoning with default settings.
- `reasoning.effort: "minimal" | "low" | "medium" | "high" | "xhigh" | "max"` controls reasoning budget where supported.
- `reasoning.effort: "none"` disables reasoning where supported.
- `reasoning.exclude: True` lets the model reason internally but hides reasoning tokens from the response.
- Returned reasoning appears in `choices[].message.reasoning` or `choices[].message.reasoning_details`.

OpenRouter docs: [reasoning tokens](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens), [API parameters](https://www.openrouter.ai/docs/api/reference/parameters).

**Reflection Prompts**
- Does explicit reasoning improve the scientific caution or just make the response longer/slower?
- Compare `reasoning_tokens`, latency, and final-answer quality.
- Decide when reasoning is worth the extra cost for biological interpretation tasks.


In [ ]:
question = "A sample has high mitochondrial RNA, low detected genes, and elevated IFIT1. Is this a dying-cell population or interferon response? Give a cautious interpretation."
reasoning_runs = [
    {
        "mode": "reasoning on (high effort)",
        "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
        "messages": [{"role": "user", "content": question}],
        "extra_body": {
            "reasoning": {
                "enabled": True,
                "effort": "xhigh",   # minimal, low, medium, high, xhigh, max
                "exclude": False,     # return reasoning if the model/provider exposes it
            }
        },
    },
    {
        "mode": "reasoning off / final answer only",
        "model": "nvidia/nemotron-nano-9b-v2:free",
        "messages": [
            {"role": "system", "content": "Answer directly. Do not include intermediate reasoning."},
            {"role": "user", "content": question},
        ],
        "extra_body": {
            "reasoning": {
                "effort": "none",
                "exclude": True,
            }
        },
    },
]

for spec in reasoning_runs:
    heading = spec["mode"]
    chunks = []
    reasoning_chunks = []
    reasoning_details = []
    usage = None
    start = time.perf_counter()
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`"])
    answer_display = display(Markdown(markdown_text), display_id=True)

    for chunk in OPENROUTER_CLIENT.chat.completions.create(
        model=spec["model"],
        messages=spec["messages"],
        temperature=0,
        extra_body=spec["extra_body"],
        stream=True,
        stream_options={"include_usage": True},
    ):
        if chunk.choices:
            delta = chunk.choices[0].delta
            delta_dict = delta.model_dump()
            if delta.content:
                chunks.append(delta.content)
                markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
                answer_display.update(Markdown(markdown_text))
            if delta_dict.get("reasoning"):
                reasoning_chunks.append(delta_dict["reasoning"])
            if delta_dict.get("reasoning_details"):
                reasoning_details.append(delta_dict["reasoning_details"])
        if chunk.usage:
            usage = chunk.usage

    completion_tokens_details = usage.completion_tokens_details if usage else None
    reasoning = "".join(reasoning_chunks) or None
    markdown_text = chr(10).join([f"### {heading}", f"`{spec['model']}`", "", "".join(chunks)])
    answer_display.update(Markdown(markdown_text))
    print("reasoning:", reasoning)
    print("reasoning_details:", reasoning_details or None)
    print("seconds:", round(time.perf_counter() - start, 3))
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
    print("reasoning_tokens:", getattr(completion_tokens_details, "reasoning_tokens", None))
    print("cost:", getattr(usage, "cost", None))
    time.sleep(REQUEST_PAUSE_SECONDS)


### reasoning on
`nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free`

**Cautious take‑away**

| Observation | What it often signals | Why it can be ambiguous |
|-------------|----------------------|--------------------------|
| **High mitochondrial RNA** | • Cells that are losing membrane integrity (apoptosis/necrosis) → mitochondrial contents leak into the cytosol and are captured in the RNA prep.<br>• Cells under severe metabolic stress (e.g., hypoxia, energy failure) that cause a burst of mitochondrial transcription. | • In many single‑cell or bulk RNA‑seq pipelines, mitochondrial transcripts are *over‑represented* simply because they are abundant, not because the cell is dead.<br>• Some healthy, activated cells (e.g., immune cells) also show elevated mt‑RNA as part of a stress‑response program. |
| **Low detected gene count** | • Overall low transcriptional activity – consistent with a cell that is dying or has shut down transcription.<br>• Technical dropout (low library complexity, poor cell quality, 3′‑bias, etc.). | • In low‑quality single‑cell libraries, many genes drop out even in healthy cells, making the “low gene count” a technical artifact rather than a biological state. |
| **Elevated IFIT1** | • Classic interferon‑stimulated gene (ISG); its up‑regulation marks an interferon response, usually in virally infected or cytokine‑treated cells. | • IFIT1 can be induced by other stress pathways (e.g., DNA damage, oxidative stress) and can also be up‑regulated in dying cells that release nucleic acids that mimic viral patterns.<br>• Some cell‑death modalities (e.g., necroptosis) are known to trigger an interferon‑type signature. |

### Putting it together

1. **Dying‑cell signature** – The combination of abundant mitochondrial RNA and a relatively low number of detected genes fits the profile of cells that are undergoing loss of homeostasis (membrane permeabilization, transcriptional shutdown). In such cells, mitochondrial transcripts can dominate the RNA pool, and the overall gene‑count drop reflects the collapse of normal transcriptional programs.

2. **Interferon‑response signature** – IFIT1 is a highly specific marker of an interferon‑driven state. Its elevation tells us that the cell (or a subset of cells) is sensing a signal that activates the IFN‑JAK‑STAT pathway, regardless of whether the cell is dying or alive.

3. **Non‑exclusive nature** – These two signals are **not mutually exclusive**. A cell that is dying can simultaneously exhibit an interferon‑type response, especially if the death is accompanied by the release of intracellular nucleic acids (e.g., mtRNA, DNA) that are sensed by pattern‑recognition receptors (RIG‑I, MDA5, cGAS). Conversely, a cell undergoing a robust IFN response may experience mitochondrial stress that raises mt‑RNA levels, and the technical quality of the RNA (e.g., from a fragile, dying cell) can depress the number of genes detected.

### What a cautious interpretation looks like

> **The data are most consistent with a mixed or transitional state: a population that is both under an interferon‑driven activation and showing signs of cellular stress or death.**  
> The high mitochondrial RNA and low gene‑count suggest compromised cell integrity, while the IFIT1 up‑regulation points to an active interferon response. Because single‑cell or bulk RNA‑seq can be confounded by technical dropout and because mitochondrial transcripts are inherently abundant, we cannot definitively label the sample as “purely dying cells” or “purely interferon‑responsive.”  

### How to move forward (if you need a more decisive answer)

| Next step | Why it helps |
|-----------|--------------|
| **Measure protein‑level markers** (e.g., caspase‑3, TUNEL, Annexin V, phospho‑STAT1) | Confirms whether the cells are truly in a death program versus merely showing IFN‑induced transcriptional changes. |
| **Assess other ISGs** (e.g., MX1, ISG15, OAS) | A broader interferon signature strengthens the IFN‑response claim and helps distinguish it from a generic stress response. |
| **Check for viral reads or dsRNA** | Direct evidence of a viral trigger would make the IFN signature more credible. |
| **Compare mitochondrial RNA to total RNA** (e.g., % of reads mapping to mitochondria) | A very high % (>30‑40 %) is typical of dying/low‑quality cells; lower percentages may indicate selective mitochondrial transcription rather than general degradation. |
| **Re‑run the analysis with stringent filtering** (e.g., minimum genes per cell, mitochondrial‑read threshold) | Reduces the chance that technical dropout is driving the low gene count. |
| **Spatial or morphological context** (e.g., imaging, histology) | Visual confirmation of cell morphology (condensed nuclei, blebbing) can corroborate a dying‑cell phenotype. |

### Bottom line

- **High mt‑RNA + low gene count** → likely a stressed or dying cell population.  
- **Elevated IFIT1** → indicates an interferon‑stimulated state.  

Because both phenomena can coexist, the safest conclusion is **“the sample shows a mixed signature: there is evidence of cellular stress/damage together with an interferon response.”** Further orthogonal data (protein markers, additional ISGs, quality controls) are needed to tease apart the two possibilities.

reasoning: We need to interpret a sample with high mitochondrial RNA, low detected genes, and elevated IFIT1. The question: "Is this a dying-cell population or interferon response? Give a cautious interpretation."

We need to consider that high mitochondrial RNA could indicate cell death (mitochondrial leakage, RNA from mitochondria) or could be due to mitochondrial stress. Low detected genes suggests low overall transcriptional activity, maybe due to cell death or low RNA capture. Elevated IFIT1 is a marker of interferon response, indicating antiviral state.

Thus we need to cautiously interpret: could be dying cells releasing mitochondrial RNA, but also could be cells undergoing interferon response with some mitochondrial involvement. The low gene detection may reflect low overall RNA quality, possibly from dying cells. IFIT1 elevation suggests interferon signaling, which can occur in dying cells (e.g., apoptosis/necrosis) or in response to viral infection.

We need to give a cautiou

### reasoning off / final answer only
`nvidia/nemotron-nano-9b-v2:free`



The elevated IFIT1 strongly suggests an interferon response, while high mitochondrial RNA and low gene detection may indicate cell death. However, these markers can overlap in scenarios where interferon signaling contributes to cell death. A cautious interpretation is that the data points to an interferon response, possibly associated with cell death, but further context is needed to distinguish the primary cause.


reasoning: None
reasoning_details: None
seconds: 34.753
prompt_tokens: 53
completion_tokens: 1017
total_tokens: 1070
reasoning_tokens: 1211
cost: 0


## 2. Temperature

Temperature controls sampling randomness: lower values produce more deterministic answers while higher values increase creativity (can also increase hallucination risk). This works by flattening the output probabilities of the model. 

We show stochasticity by repeating generating the answers multiple times. The key comparison is not only temperature 0 versus 1, but also variation across repeated runs at the same temperature. At temperature 0, repeated answers should be relatively stable; at temperature 1, the selected experiments and framing should vary more.

**Reflection Prompts**
- Compare replicates at the same temperature: which elements remain stable, and which ones change?
- Evaluate whether greater variety produces genuinely more useful ideas or just different wording.
- Decide which temperature you would use for a scientific task and justify the trade-off between creativity and control.


In [ ]:
question = """
The airway bulk RNA-seq experiment identified genes that change expression
after dexamethasone treatment.

Suggest three substantially different follow-up experiments.
Answer in concise bullet points, max 120 words.
"""
rows = []

for temperature in [0, 1.0]:
    for replicate in range(2):
        start = time.perf_counter()

        response = OPENROUTER_CLIENT.chat.completions.create(
            model=DEFAULT_MODEL_ID,
            messages=[{"role": "user", "content": question}],
            temperature=temperature,
        )
        usage = response.usage
        completion_tokens_details = usage.completion_tokens_details
        answer = response.choices[0].message.content
        display(Markdown(f"### temperature={temperature}, replicate={replicate + 1}\n\n{answer}"))

        rows.append({
            "temperature": temperature,
            "replicate": replicate + 1,
            "seconds": round(time.perf_counter() - start, 3),
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "reasoning_tokens": completion_tokens_details.reasoning_tokens,
            "cost": usage.cost,
            "usage": usage,
            "answer": answer,
        })
        time.sleep(REQUEST_PAUSE_SECONDS)
        
df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 3. System prompt

The system prompt sets high-level behavior such as persona, assumption, criteria for tool usage, and any instruction that should always be followed. This is injected at the first turn of the conversation and is kept in memory across all turns. Implemented via `SystemMessage` inside a `ChatPromptTemplate`. 

**Reflection Prompts**
- Observe how the tone changes when the model receives a role or more specific instructions.
- Identify whether the system prompt improves accuracy as well, or mainly changes the form of the answer.
- Ask which instructions make the answer easier to evaluate from a scientific perspective.


In [ ]:
question = "A sample has high mitochondrial RNA and low detected genes. Is it a dying-cell population?"
rows = []

# System prompt: minimal
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "minimal", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# System prompt: biologist persona
messages = [
    {"role": "system", "content": "You are a molecular biologist who explains concepts clearly to computational biology students."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "biologist persona", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# System prompt: scientific with strict instructions
messages = [
    {"role": "system", "content": "You are a strict scientific assistant. Separate evidence from speculation, state uncertainty, and avoid unsupported claims."},
    {"role": "user", "content": question},
]
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=messages,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"system_prompt": "strict scientific", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 4. Prompt template

This is a scaffold that controls how the task is framed and allows the injection of dynamic data like user inputs or variables at runtime. Implemented via `PromptTemplate` or `ChatPromptTemplate`. 

**Reflection Prompts**
- Compare how strongly the prompt structure guides the structure of the final answer.
- Notice whether the few-shot examples help the model imitate the format or also reason better.
- Identify which template makes it easiest to correct or compare answers across groups.


In [ ]:
observation = "a T-cell cluster has high interferon-stimulated genes after stimulation"
rows = []

# Free form prompt:  a plain instruction with no examples or required output structure
prompt = f"Explain whether {observation} is biologically meaningful."
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "free form", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# Few-shot prompt: includes an example input-output pair to guide the answer style
prompt = (
    "Example:\n"
    "Observation: high MALAT1 in low-quality nuclei.\n"
    "Answer: This may reflect nuclear RNA content or technical quality; validate with QC metrics and markers.\n\n"
    f"Observation: {observation}\n"
    "Answer:"
)
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "few shot", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
time.sleep(REQUEST_PAUSE_SECONDS)

# Structured prompt: asks the model to organize its answer into named sections
prompt = (
    f"Observation: {observation}\n"
    "Return sections: Interpretation | Alternative explanations | Checks | Confidence."
)
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
rows.append({"template": "structured", "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 5. Context length

LLM answers depend on what is present in the prompt context window. Models with large context windows receive more background information, which can improve grounding but can also introduce distractions, truncation, and irrelevant details. Context is an intrinsic characteristic of an LLM, and should be taken into account when choosing an LLM for your specific purposes. 


**Reflection Prompts**
- Evaluate which information is preserved when the context is short and which only appears with more context.
- Look for signs of distraction: does more context always make the answer better?
- Discuss what minimum context would be sufficient to answer the question responsibly.


In [ ]:
base_context = "Protocol note: Samples were PBMCs stimulated with IFN-beta for 6 hours. Mitochondrial reads above 20% were filtered."
small_context = base_context
large_context = "\n".join([base_context] + [
    "Marker note: IFIT1, ISG15, MX1, and OAS1 indicate interferon response.",
    "QC note: doublet scores above 0.25 were removed.",
    "Batch note: donor and library chemistry can confound differential expression.",
] * 8)
too_much_context = large_context + "\n" + "\n".join(
    [f"Irrelevant lab inventory line {i}: freezer box metadata unrelated to expression." for i in range(120)]
)

contexts = {"no_context": "", "small_context": small_context, "large_context": large_context, "too_much_context": too_much_context}
question = "Why might IFIT1 and ISG15 be elevated, and what caveats should be checked?"
rows = []
for label, ctx in contexts.items():
    prompt = f"Context:\n{ctx}\n\nQuestion: {question}" if ctx else question
    start = time.perf_counter()
    response = OPENROUTER_CLIENT.chat.completions.create(
        model=DEFAULT_MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    usage = response.usage
    completion_tokens_details = usage.completion_tokens_details
    rows.append({"context": label, "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": response.choices[0].message.content})
    time.sleep(REQUEST_PAUSE_SECONDS)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Output parsing

This determines in what format the LLM produces the output, eg. plain text, JSON, pydantic. This is relevant in the context of agents because downstream processes that use LLM's output as input may require it in specific formats. Additionally, structured formats enforce strict validation checking.  

**Reflection Prompts**
- Compare free-form and structured output: which is easier to read, validate, and reuse?
- Observe what is lost when a rich answer has to fit into predefined fields.
- Decide when it is worth enforcing a rigid schema in a scientific pipeline.


In [ ]:
QUESTION = "Interpret high FKBP5 and CRISPLD2 expression in dexamethasone-treated airway smooth muscle cells."


### i. Free text

Free text output is easy for humans to read but unreliable for software to process downstream.

**Reflection Prompts**
- Identify which parts of the answer are clear for humans but difficult to compare automatically.
- Note whether the model signals uncertainty or caveats without being forced to do so by a schema.
- Think about which criteria you would use to assign consistent scores to free-form answers.


In [ ]:
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": QUESTION}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
answer = response.choices[0].message.content
display(Markdown(answer))

df = pd.DataFrame([{
    "format": "free text",
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": usage.prompt_tokens,
    "completion_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "reasoning_tokens": completion_tokens_details.reasoning_tokens,
    "cost": usage.cost,
    "usage": usage,
    "answer": answer,
}])
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### ii. JSON output

JSON output asks the model to return machine-readable fields. However, malformed JSON can still occur without validation or native structured output.


**Reflection Prompts**
- Check whether the JSON is valid and whether all fields are filled in informatively.
- Compare human readability with usefulness for an automated pipeline.
- Look for examples where the model follows the format but oversimplifies the content.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser

parser = JsonOutputParser()
prompt = f"Return valid JSON only with keys answer, confidence, caveats.\n\nQuestion: {QUESTION}"
start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
parsed_response = parser.parse(response.choices[0].message.content)

df = pd.DataFrame([{
    "format": "JSON",
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": usage.prompt_tokens,
    "completion_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "reasoning_tokens": completion_tokens_details.reasoning_tokens,
    "cost": usage.cost,
    "usage": usage,
    "parsed_response": parsed_response,
    "answer": response.choices[0].message.content,
}])
df.style.set_properties(
    subset=["answer", "parsed_response", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### iii. Pydantic parser

Pydantic parser validates types and required fields, making outputs usable directly in downstream Python code.

In this example, we can access the `parser.answer`, `parser.confidence`, and `parser.caveats` variables within Python, making it much easier to build reliable pipelines and AI agents. 


**Reflection Prompts**
- Observe which schema constraints help catch errors or ambiguity.
- Evaluate whether validation improves scientific quality or only formal compliance.
- Discuss which fields you would add to make the answer more useful in a lab setting.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BioAnswer(BaseModel):
    answer: str
    confidence: Literal["low", "medium", "high"]
    caveats: list[str]

parser = PydanticOutputParser(pydantic_object=BioAnswer)

prompt = f"""
You are assisting with RNA-seq interpretation.
{parser.get_format_instructions()}
Question:
{QUESTION}
"""

start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
parsed_response = parser.parse(response.choices[0].message.content)
print(parsed_response)

df = pd.DataFrame([{
    "format": "Pydantic",
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": usage.prompt_tokens,
    "completion_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "reasoning_tokens": completion_tokens_details.reasoning_tokens,
    "cost": usage.cost,
    "usage": usage,
    "parsed_response": parsed_response,
    "answer": response.choices[0].message.content,
}])
df.style.set_properties(
    subset=["answer", "parsed_response", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


Downstream use of pydantic parsing: routing the pipeline/agent based on confidence:

In [ ]:
if parsed_response.confidence == "high":
    print("✅ Proceed with pathway enrichment.")
elif parsed_response.confidence == "medium":
    print("⚠️ Check supporting literature before interpreting.")
else:
    print("❌ Collect more evidence before drawing conclusions.")

### vi. Markdown formatting

Markdown formatting controls the shape of the answer, such as plain text, tables, or bullet lists, which affects readability and downstream copy-paste usability.


**Reflection Prompts**
- Compare which format makes it easiest to quickly find results, caveats, and conclusions.
- Evaluate whether the table forces a more orderly comparison than the paragraph or bullet points.
- Ask which format you would use for personal notes, scientific reports, or machine-readable output.


In [ ]:
formats = {
    "plain_text": "Answer in one short paragraph.",
    "table": "Answer as a markdown table with columns Finding, Interpretation, Caveat.",
    "bullet_list": "Answer as concise bullet points.",
}
question = "Summarize how to interpret marker genes, QC metrics, and batch effects in scRNA-seq."
rows = []
for label, instruction in formats.items():
    prompt = f"{instruction}\n\n{question}"
    start = time.perf_counter()
    response = OPENROUTER_CLIENT.chat.completions.create(
        model=DEFAULT_MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    usage = response.usage
    completion_tokens_details = usage.completion_tokens_details
    answer = response.choices[0].message.content
    display(Markdown(f"### {label}\n\n{answer}"))
    rows.append({"format": label, "seconds": round(time.perf_counter() - start, 3), "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens, "total_tokens": usage.total_tokens, "reasoning_tokens": completion_tokens_details.reasoning_tokens, "cost": usage.cost, "usage": usage, "answer": answer})
    time.sleep(REQUEST_PAUSE_SECONDS)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 7. Streaming

Streaming controls whether tokens are delivered incrementally, which affects user experience and perceived latency without necessarily changing the final answer.


**Reflection Prompts**
- Distinguish perceived latency from total time: which matters more for the final user?
- Observe whether seeing the answer as it is generated changes how you evaluate it.
- Think of scientific cases where streaming is useful and cases where it could be distracting.


In [ ]:
prompt = "Give a concise explanation of pseudobulk differential expression."

chunks = []
stream_usage = None
start = time.perf_counter()
for chunk in OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    stream=True,
    stream_options={"include_usage": True},
):
    if chunk.choices and chunk.choices[0].delta.content:
        text = chunk.choices[0].delta.content
        chunks.append(text)
        print(text, end="")
    if chunk.usage:
        stream_usage = chunk.usage

answer = "".join(chunks)
usage = stream_usage
completion_tokens_details = usage.completion_tokens_details if usage else None
print("\n\nstream_seconds:", round(time.perf_counter() - start, 3))

stream_df = pd.DataFrame([{
    "stream": True,
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": getattr(usage, "prompt_tokens", None),
    "completion_tokens": getattr(usage, "completion_tokens", None),
    "total_tokens": getattr(usage, "total_tokens", None),
    "reasoning_tokens": getattr(completion_tokens_details, "reasoning_tokens", None),
    "cost": getattr(usage, "cost", None),
    "usage": usage,
    "answer": answer,
}])
stream_df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 8. Citation generation

Citation generation asks the model to attach source references to claims, which improves auditability only when the citations are grounded in provided sources or retrieval metadata.


**Reflection Prompts**
- Check whether each important claim is linked to the correct source.
- Distinguish genuinely supported citations from citations added only as decoration.
- Ask which checks would be needed before trusting generated citations.


In [ ]:
context = (
    "[S1] IFIT1, IFIT3, ISG15, MX1, and OAS genes are common interferon-stimulated genes.\n"
    "[S2] High mitochondrial RNA fractions can indicate low-quality or stressed cells, but thresholds are tissue and protocol dependent."
)
prompt = f"Use only the sources below and cite each claim with [S1] or [S2].\n\n{context}\n\nQuestion: Interpret high IFIT1 and high mitochondrial RNA."

start = time.perf_counter()
response = OPENROUTER_CLIENT.chat.completions.create(
    model=DEFAULT_MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=ANSWER_MAX_TOKENS,
    temperature=0,
)
usage = response.usage
completion_tokens_details = usage.completion_tokens_details
answer = response.choices[0].message.content
display(Markdown(answer))

df = pd.DataFrame([{
    "format": "cited answer",
    "seconds": round(time.perf_counter() - start, 3),
    "prompt_tokens": usage.prompt_tokens,
    "completion_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "reasoning_tokens": completion_tokens_details.reasoning_tokens,
    "cost": usage.cost,
    "usage": usage,
    "answer": answer,
}])
df.style.set_properties(
    subset=["answer", "usage"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)
